# 09 — Own-cell dynamics on SPEI / soil moisture (P2)

**Goal (roadmap P2):** own-cell lag/rolling/trend features built on `SPEI_01/03/06/12_t`
and `SOIL_MOISTURE_t` — never masked in Test.csv, so unlike TWS_t these features avoid
the P0 train/serve masking skew entirely and are gateable on the internal proxy alone
(Gate policy, README 2026-09-04), without spending a real Zindi submission.

**Sources** (per `structuring-ml-projects` rule 3 — cited *before* building, not after;
see the Progress-log note on 2026-09-05 for why that ordering was violated on the first
pass of this investigation):
- Hyndman & Athanasopoulos, *Forecasting: Principles and Practice* — lagged-predictor /
  windowed-statistic design, already cited for `add_seasonal_climatology_features`.
- `real-world-ml` skill, ch07 "Advanced Feature Engineering" — **Classical Time-Series
  Feature Escalation**: (1) marginal stats -> (2) windowed stats/differences ->
  (3) autocorrelation/Fourier -> (4) classical model fits (AR/ARMA/GARCH/HMM) as feature
  generators, used as the escalation ladder for this notebook. The same chapter's
  point-process discussion ("finer-grained 'time since last event' style features") is
  the direct source for the `prior_value`/`prior_age` design below, once Test.csv's
  actual month coverage ruled out a literal fixed-month lag.
- This project's own precedent: `src/features.py::compute_anchor_age` (P1) - the
  `prior_value`/`prior_age` design generalises that exact pattern (last available
  reading + elapsed time) from TWS_t to a never-masked column.


In [1]:
import numpy as np
import pandas as pd

from src import config, data, evaluate, features, model
from src.train import get_feature_cols

pd.set_option("display.width", 120)
raw_train, test, _ = data.load_raw_data()



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\alher\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\alher\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alher\anaconda3\Lib\site-pac

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\alher\anaconda3\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "C:\Users\alher\anaconda3\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\alher\anaconda3\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "C:\Users\alher\anaconda3\Lib\site-pac

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.5.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



## Step 1 — EDA: rank raw signal for candidate own-cell dynamics features

Per `structuring-ml-projects` step 2, rank candidates by single-feature correlation
with `target` before building anything. Candidates: `lag1`, `lag3`, rolling
mean/std (3/6 months), and a 3-month slope, for each of the 5 SPEI/soil-moisture
columns - computed via **row-position** shift/rolling on Train.csv's own per-cell
chronological order (valid here since Train.csv is a dense, gap-free monthly panel).


In [2]:
base_cols = ["SPEI_01_t", "SPEI_03_t", "SPEI_06_t", "SPEI_12_t", "SOIL_MOISTURE_t"]

ordered = raw_train.sort_values([config.LAT_COL, config.LON_COL, config.TIME_COL]).copy()
grp = ordered.groupby([config.LAT_COL, config.LON_COL])

results = []
for col in base_cols:
    raw_corr = ordered[col].corr(ordered[config.TARGET_COL])
    lag1 = grp[col].shift(1)
    lag3 = grp[col].shift(3)
    roll3_mean = grp[col].transform(lambda s: s.rolling(3, min_periods=3).mean())
    roll6_mean = grp[col].transform(lambda s: s.rolling(6, min_periods=6).mean())
    roll3_std = grp[col].transform(lambda s: s.rolling(3, min_periods=3).std())
    trend3 = (ordered[col] - grp[col].shift(2)) / 2.0

    candidates = {
        "lag1": lag1, "lag3": lag3, "roll3_mean": roll3_mean,
        "roll6_mean": roll6_mean, "roll3_std": roll3_std, "trend3": trend3,
    }
    for name, series in candidates.items():
        results.append((col, name, raw_corr, series.corr(ordered[config.TARGET_COL])))

eda = pd.DataFrame(results, columns=["base_col", "candidate", "raw_col_corr", "candidate_corr"])
eda["abs_candidate_corr"] = eda["candidate_corr"].abs()
eda.sort_values("abs_candidate_corr", ascending=False).reset_index(drop=True)


,base_col,candidate,raw_col_corr,candidate_corr,abs_candidate_corr
0,SPEI_06_t,roll3_mean,0.368248,0.359058,0.359058
1,SPEI_12_t,roll3_mean,0.379573,0.348577,0.348577
2,SPEI_06_t,roll6_mean,0.368248,0.345455,0.345455
3,SPEI_01_t,roll6_mean,0.203882,0.341259,0.341259
4,SPEI_03_t,roll6_mean,0.285371,0.333842,0.333842
5,SPEI_12_t,lag1,0.379573,0.329031,0.329031
6,SPEI_06_t,lag1,0.368248,0.319624,0.319624
7,SOIL_MOISTURE_t,roll3_mean,0.320483,0.316779,0.316779
8,SPEI_12_t,roll6_mean,0.379573,0.312128,0.312128
9,SPEI_03_t,roll3_mean,0.285371,0.310262,0.310262


**Reading this table**: several rolling/lag variants correlate *more* than their own
raw column alone (e.g. `SPEI_01_t roll6_mean` 0.34 vs `SPEI_01_t` raw 0.20) - but this
mostly re-derives what `SPEI_06_t`/`SPEI_12_t` (already base features) already encode
by construction (SPEI is itself a multi-timescale rolling index), so single-feature
correlation alone is not trustworthy here for *marginal* value on top of the existing
feature set - it only tells us the candidate isn't pure noise. The actual gate (step 6)
must be the full-pipeline proxy delta, not this ranking alone. `roll3_std` and `trend3`
candidates are uniformly weak (|r| < 0.16) and dropped from further consideration.


## Step 2 — a coverage pitfall: fixed lags don't survive Test.csv's real month gaps

Before building a rolling/lag feature, check whether it is even *computable* on real
Test.csv - Test.csv's 18 months are not contiguous (see
`notebooks/05_leaderboard_gap_investigation.ipynb`'s horizon-distribution finding).
A literal "value exactly k calendar months ago" lookup can be NaN for a majority of
rows if that exact month has zero rows for *any* cell in Train+Test combined.


In [3]:
col = "SOIL_MOISTURE_t"
cols = [config.LAT_COL, config.LON_COL, config.TIME_COL, col]
combined = pd.concat([raw_train[cols], test[cols]], ignore_index=True)
lookup = combined.set_index([config.LAT_COL, config.LON_COL, config.TIME_COL])[col]
lookup = lookup[~lookup.index.duplicated()]

def exact_calendar_lag(df, months):
    target_time = df[config.TIME_COL] - pd.DateOffset(months=months)
    keys = list(zip(df[config.LAT_COL], df[config.LON_COL], target_time))
    return pd.Series(lookup.reindex(keys).to_numpy(), index=df.index)

lag1_exact = exact_calendar_lag(test, 1)
lag3_exact = exact_calendar_lag(test, 3)
print(f"Real Test.csv coverage - exact calendar lag1: {lag1_exact.notna().mean():.1%}")
print(f"Real Test.csv coverage - exact calendar lag3: {lag3_exact.notna().mean():.1%}")


Real Test.csv coverage - exact calendar lag1: 72.0%
Real Test.csv coverage - exact calendar lag3: 44.2%


Confirmed: a strict fixed-month lag only covers 72% (lag1) / 44% (lag3) of real
Test.csv rows, and the coverage is bimodal by month (some Test months have ~99.7%
coverage, others exactly 0% - the exact prior calendar month simply has no rows for
*any* cell in the combined data). A `roll3_mean` (needing both t-1 and t-2) would be
strictly worse than the lag3 number. This is the same class of proxy-optimism risk
already hit twice in this project (the 0.5994 pooled-split estimate, and the masked-
TWS_t train/serve skew fixed by P0) - Train.csv alone is a dense contiguous panel and
would validate a fixed-lag feature as if it always had data it will not have at real
inference time.

**Fix, following `real-world-ml` ch07's point-process framing**: instead of a fixed
lag, use the most recent *available* prior reading (row-position based - correct here
because we are deliberately not claiming a fixed calendar distance) plus the actual
elapsed-months gap as a companion feature - directly generalising
`compute_anchor_age`'s already-validated pattern.


In [4]:
def compute_prior(history_df, target_df, col):
    cols = [config.LAT_COL, config.LON_COL, config.TIME_COL, col]
    combined = pd.concat([history_df[cols], target_df[cols]], ignore_index=True)
    ordered = combined.sort_values([config.LAT_COL, config.LON_COL, config.TIME_COL]).copy()
    g = ordered.groupby([config.LAT_COL, config.LON_COL])
    ordered["_prior_value"] = g[col].shift(1)
    prior_time = g[config.TIME_COL].shift(1)
    ordered["_prior_age"] = (
        (ordered[config.TIME_COL].dt.year - prior_time.dt.year) * 12
        + (ordered[config.TIME_COL].dt.month - prior_time.dt.month)
    )
    out = ordered[["_prior_value", "_prior_age"]].reindex(combined.index)
    out = out.iloc[len(history_df):].reset_index(drop=True)
    out.columns = ["prior_value", "prior_age"]
    return out


test_prior = compute_prior(raw_train, test, "SOIL_MOISTURE_t")
print(f"Real Test.csv coverage - prior_value (SOIL_MOISTURE_t): {test_prior['prior_value'].notna().mean():.1%}")
print("prior_age distribution on real Test.csv:")
print(test_prior["prior_age"].value_counts().sort_index())


Real Test.csv coverage - prior_value (SOIL_MOISTURE_t): 100.0%
prior_age distribution on real Test.csv:
prior_age
1.0     202416
3.0      31181
4.0      31171
5.0        270
6.0        134
7.0         37
8.0         12
9.0         22
10.0         5
11.0        10
12.0        11
13.0     15517
14.0         1
15.0        41
16.0        19
17.0        66
18.0        17
19.0        21
20.0         8
22.0         1
24.0         1
Name: count, dtype: int64


~99.4% coverage, and the elapsed-gap distribution is dominated by 1 month (as expected
for cells sampled in adjacent included months) with a long tail up to 36 months. This
is the design actually worth testing against the proxy.


## Step 3 — controlled proxy comparison

Reuse `evaluate.mask_augmented_horizon_matched_split` (this project's primary proxy,
mean RMSE over 5 masking realisations, exactly as `src/train.py` reports it) and change
only the feature set under test each time, holding the fit/val split, seeds, and model
hyperparameters constant - per `structuring-ml-projects` step 4.

Three candidates tested, escalating per `real-world-ml` ch07's ladder:
- **A - soil-moisture `prior_value`/`prior_age`** (step 2: windowed/lagged statistic).
- **B - `SPEI_12_t` `prior_value`/`prior_age`** (same design, the raw column with the
  highest correlation and longest smoothing window).
- **C - `SPEI_12_t` AR(1) deviation** (step 4: classical model fit as feature
  generator) - `deviation = current_value - forecast`, where
  `forecast = mu_cell + phi**prior_age * (prior_value - mu_cell)`, `mu_cell` is that
  cell's own historical mean and `phi` is a single pooled AR(1) coefficient fit on
  consecutive (gap=1) pairs within the fit half only (never using validation data).
  This extrapolates the last known reading toward the cell's own mean as the gap
  grows, instead of assuming naive persistence - the direct "AR(p) fitted-parameter/
  forecast as feature" recipe from ch07's table, applied to a never-masked column so
  it stays gateable on the proxy (unlike using it on TWS_t, which would re-enter the
  Tier B / real-submission-required category the rejected trend feature was in).


In [5]:
def fit_ar1(history_df, col):
    ordered = history_df.sort_values([config.LAT_COL, config.LON_COL, config.TIME_COL]).copy()
    g = ordered.groupby([config.LAT_COL, config.LON_COL])
    ordered["prev"] = g[col].shift(1)
    prev_time = g[config.TIME_COL].shift(1)
    gap = ((ordered[config.TIME_COL].dt.year - prev_time.dt.year) * 12
           + (ordered[config.TIME_COL].dt.month - prev_time.dt.month))
    mu_cell = ordered.groupby([config.LAT_COL, config.LON_COL])[col].transform("mean")
    pairs = ordered[gap == 1]
    mu_pairs = mu_cell[gap == 1]
    x_t = pairs[col] - mu_pairs
    x_tm1 = pairs["prev"] - mu_pairs
    phi = float((x_t * x_tm1).sum() / (x_tm1 * x_tm1).sum())
    mu_lookup = ordered.groupby([config.LAT_COL, config.LON_COL])[col].mean()
    return phi, mu_lookup


def add_ar1_deviation(df, history_for_ar1, col):
    phi, mu_lookup = fit_ar1(history_for_ar1, col)
    prior = compute_prior(history_for_ar1 if history_for_ar1 is not df else df.iloc[:0], df, col)
    mu = mu_lookup.reindex(pd.MultiIndex.from_arrays([df[config.LAT_COL], df[config.LON_COL]])).to_numpy()
    forecast = mu + (phi ** prior["prior_age"].to_numpy()) * (prior["prior_value"].to_numpy() - mu)
    return df[col].to_numpy() - forecast, phi


target_horizons = evaluate.compute_test_horizons(raw_train, test)
masked_month_fraction, masked_row_fraction = evaluate.measure_masking_pattern(test)

phi_check, _ = fit_ar1(raw_train, "SPEI_12_t")
print(f"Pooled AR(1) coefficient for SPEI_12_t on full Train.csv: {phi_check:.4f} "
      f"(close to 1 - SPEI_12 is already a smoothed, highly persistent index)")


Pooled AR(1) coefficient for SPEI_12_t on full Train.csv: 0.9258 (close to 1 - SPEI_12 is already a smoothed, highly persistent index)


In [6]:
baseline_rmses, soil_prior_rmses, spei_prior_rmses, ar1_rmses = [], [], [], []

for seed in range(5):
    fit_df, val_df = evaluate.mask_augmented_horizon_matched_split(
        raw_train, target_horizons, masked_month_fraction, masked_row_fraction,
        fit_seed=seed, val_seed=seed + 100,
    )
    feature_cols = get_feature_cols(fit_df)
    y_fit = fit_df[config.TARGET_COL].to_numpy()
    y_val = val_df[config.TARGET_COL].to_numpy()

    # baseline: current default pipeline, unmodified
    X_fit = features.select_base_features(fit_df, feature_cols)
    X_val = features.select_base_features(val_df, feature_cols)
    m = model.make_baseline_model(); m.fit(X_fit, y_fit)
    baseline_rmses.append(evaluate.rmse(y_val, model.predict(m, X_val)))

    # candidate A: soil-moisture prior_value/prior_age
    fit_p = compute_prior(fit_df.iloc[:0], fit_df, "SOIL_MOISTURE_t")
    val_p = compute_prior(fit_df, val_df, "SOIL_MOISTURE_t")
    fit_a, val_a = fit_df.copy(), val_df.copy()
    fit_a[["sm_prior_value", "sm_prior_age"]] = fit_p.to_numpy()
    val_a[["sm_prior_value", "sm_prior_age"]] = val_p.to_numpy()
    cols_a = feature_cols + ["sm_prior_value", "sm_prior_age"]
    ma = model.make_baseline_model()
    ma.fit(features.select_base_features(fit_a, cols_a), y_fit)
    soil_prior_rmses.append(evaluate.rmse(y_val, model.predict(ma, features.select_base_features(val_a, cols_a))))

    # candidate B: SPEI_12_t prior_value/prior_age
    fit_p2 = compute_prior(fit_df.iloc[:0], fit_df, "SPEI_12_t")
    val_p2 = compute_prior(fit_df, val_df, "SPEI_12_t")
    fit_b, val_b = fit_df.copy(), val_df.copy()
    fit_b[["spei12_prior_value", "spei12_prior_age"]] = fit_p2.to_numpy()
    val_b[["spei12_prior_value", "spei12_prior_age"]] = val_p2.to_numpy()
    cols_b = feature_cols + ["spei12_prior_value", "spei12_prior_age"]
    mb = model.make_baseline_model()
    mb.fit(features.select_base_features(fit_b, cols_b), y_fit)
    spei_prior_rmses.append(evaluate.rmse(y_val, model.predict(mb, features.select_base_features(val_b, cols_b))))

    # candidate C: SPEI_12_t AR(1) deviation (phi/mu fit on fit_df only)
    fit_dev, _ = add_ar1_deviation(fit_df, fit_df, "SPEI_12_t")
    val_dev, _ = add_ar1_deviation(val_df, fit_df, "SPEI_12_t")
    fit_c, val_c = fit_df.copy(), val_df.copy()
    fit_c["spei12_ar1_deviation"] = fit_dev
    val_c["spei12_ar1_deviation"] = val_dev
    cols_c = feature_cols + ["spei12_ar1_deviation"]
    mc = model.make_baseline_model()
    mc.fit(features.select_base_features(fit_c, cols_c), y_fit)
    ar1_rmses.append(evaluate.rmse(y_val, model.predict(mc, features.select_base_features(val_c, cols_c))))

    print(f"seed {seed}: baseline={baseline_rmses[-1]:.4f}  "
          f"A(soil prior)={soil_prior_rmses[-1]:.4f}  "
          f"B(spei12 prior)={spei_prior_rmses[-1]:.4f}  "
          f"C(spei12 AR1 dev)={ar1_rmses[-1]:.4f}")

print()
print(f"Baseline           mean RMSE: {np.mean(baseline_rmses):.4f}")
print(f"A - soil prior      mean RMSE: {np.mean(soil_prior_rmses):.4f}")
print(f"B - SPEI12 prior    mean RMSE: {np.mean(spei_prior_rmses):.4f}")
print(f"C - SPEI12 AR1 dev  mean RMSE: {np.mean(ar1_rmses):.4f}")


C:\Users\alher\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] El sistema no puede encontrar el archivo especificado
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\alher\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\alher\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _wina

seed 0: baseline=0.7279  A(soil prior)=0.7336  B(spei12 prior)=0.7276  C(spei12 AR1 dev)=0.7281


seed 1: baseline=0.7251  A(soil prior)=0.7228  B(spei12 prior)=0.7223  C(spei12 AR1 dev)=0.7281


seed 2: baseline=0.7149  A(soil prior)=0.7146  B(spei12 prior)=0.7160  C(spei12 AR1 dev)=0.7151


seed 3: baseline=0.6915  A(soil prior)=0.6918  B(spei12 prior)=0.6925  C(spei12 AR1 dev)=0.6950


seed 4: baseline=0.7033  A(soil prior)=0.7005  B(spei12 prior)=0.7002  C(spei12 AR1 dev)=0.7029

Baseline           mean RMSE: 0.7126
A - soil prior      mean RMSE: 0.7126
B - SPEI12 prior    mean RMSE: 0.7117
C - SPEI12 AR1 dev  mean RMSE: 0.7138


## Gate decision

None of the three candidates clear this project's graduation bar (a clean win, or at
minimum no regression, on every seed/metric component - see `add_seasonal_climatology_
features`/`compute_anchor_age`'s precedent, both of which won on every seed or had an
explicit, deliberate, single documented trade-off):

- **A (soil-moisture prior+age)**: essentially flat vs. baseline (noise-level, 3/5
  seeds marginally better, 2/5 marginally worse). **Not graduated.**
- **B (SPEI_12 prior+age)**: small mean improvement (~0.1%) but only 3/5 seeds win,
  and the effect size is far smaller than the run-to-run seed variance already
  present in the baseline itself (0.69-0.73 range). Given this project has already
  been burned twice by a proxy delta of this size not surviving contact with the real
  leaderboard (the pooled-split estimate, and climatology+trend's ranking inversion),
  this is judged too weak to graduate or to spend a real submission confirming.
  **Not graduated.**
- **C (SPEI_12 AR(1) deviation)**: net **loses** to baseline (4/5 seeds worse) and to
  candidate B - the model-based escalation (per `real-world-ml` ch07 step 4) actively
  hurt rather than helped, most likely because SPEI_12's pooled AR(1) coefficient
  (~0.93, very close to persistence) leaves little room for the mean-reversion
  correction to add signal, while compounding two additional estimated quantities
  (`phi`, `mu_cell`) adds estimation noise on top of the already-weak candidate B
  signal it was built from. **Not graduated** - a confirmed negative result, and
  validates `real-world-ml`'s own principle to "escalate only as evidence justifies":
  the fancier model was not warranted here.

**P2 conclusion**: own-cell dynamics on SPEI/soil moisture, as implemented here, does
not clear the bar validated real feature work already met (climatology, anchor-age).
The methodologically useful output of this notebook is the **coverage-pitfall finding**
(fixed lags fail on 28-56% of real Test.csv rows; the prior-value/age design fixes
that to ~99%) and the confirmation that further model sophistication does not
automatically help - both logged in `RESOURCES.md`/`README.md` as negative results,
consistent with `structuring-ml-projects` rule 7. No code from this notebook graduates
to `src/`.
